# 03 — Explanation Faithfulness (D7–D9)

**This notebook is the paper.** Detection accuracy is table stakes — ≥5 YOLO-family works already exist on VinDr-CXR. What no verified prior work does is *quantify* whether the model looks where the radiologist looked.

The asset: **VinDr ships ground-truth boxes.** Most CXR-XAI work uses NIH ChestX-ray14, which has none and can therefore only show qualitative heatmaps.

> ⚠️ **The trap:** "we ran three YOLOs and made heatmaps" is a course project — literally what the vendored `tariqshaban` repo already is. The measured saliency evaluation is the whole difference.
>
> **If the schedule slips, cut a model. Never cut these metrics.**

---
### What the detection results set up

From `docs/analysis/detection-results.md` (test split, 3 seeds):

| | YOLOv8s | YOLO11s | YOLO26s |
|---|---|---|---|
| mAP@0.4 overall | **0.413** | 0.402 | 0.400 |
| **Calcification** (small) | 0.079 | 0.073 | **0.126** |
| **Nodule/Mass** (small) | 0.281 | **0.328** | **0.331** |
| Atelectasis (diffuse) | **0.310** | 0.293 | 0.219 |
| Aortic enlargement (anchored) | 0.956 | 0.950 | 0.943 |

**YOLO26 trades large-lesion performance for small-lesion performance.** That gives Axis A a sharp, falsifiable question:

> YOLO26 genuinely detects Calcification better (+60%, ~5× the per-class std). **Does it also explain it better?** If detection improves but faithfulness doesn't, that is accuracy/faithfulness decoupling demonstrated on a class where we have independent evidence the model actually improved.

And separately: Aortic enlargement and Cardiomegaly sit at **0.95** while the other twelve average **0.31**. Both are anatomically anchored — same image region in every PA radiograph. §2b asks whether the model attends to the anatomy or merely to position.

---
**Prerequisite:** Add Data → "Notebook Output Files" → "Your Work" → attach **both** notebook 01 (dataset) and notebook 02 (weights). **Accelerator:** `GPU T4 x2`, `device=0`.

In [ ]:
!pip install -q ultralytics grad-cam

REPO = "/kaggle/working/repo"
!rm -rf {REPO} && git clone -q https://github.com/ShMazumder/vinbigdata-chest-x-ray.git {REPO}

import sys
sys.path.insert(0, REPO)

from pathlib import Path
import pandas as pd, numpy as np
from ultralytics import YOLO

from src import config as C
from src.evaluation import xai
from src.utils.run_logger import capture_gpu

!cd {REPO} && git rev-parse --short HEAD
capture_gpu(strict=True)

> **Re-running after a mid-session push?** `git pull` updates the files but `import` returns the cached module — you silently run the old code. Clear it:
> ```python
> !cd /kaggle/working/repo && git pull -q
> import sys
> for m in [k for k in sys.modules if k.startswith("src")]: del sys.modules[m]
> from src import config as C
> from src.evaluation import xai
> ```

In [ ]:
DATA_YAML = C.find_data_yaml()
DATA_ROOT = C.dataset_root(DATA_YAML)   # NOT .parent -- may be a repaired yaml
TEST_IMG = DATA_ROOT / "images" / "test"
TEST_LBL = DATA_ROOT / "labels" / "test"
assert TEST_IMG.is_dir(), f"no test images at {TEST_IMG}"
print(f"test split: {len(list(TEST_IMG.glob('*.jpg')))} images")

In [ ]:
import re

runs = {}
for p in Path("/kaggle/input").glob("**/weights/best.pt"):
    name = p.parent.parent.name              # yolo26s_512_e40_seed0
    m = re.match(r"(\w+?)_\d+_e\d+_seed(\d+)$", name)
    if m and m.group(1) in C.MODELS:
        runs[(m.group(1), int(m.group(2)))] = str(p)

assert runs, "attach notebook 02's output"
MODEL_KEYS = sorted({k for k, _ in runs})
SEEDS_FOUND = sorted({s for _, s in runs})
print(f"{len(runs)} runs | models {MODEL_KEYS} | seeds {SEEDS_FOUND}")

## 1. Target layer — GATE, do this first

> [!CAUTION]
> **The first implementation of this was wrong on all three models, in different ways.** It took the last `Conv2d` in the network, which resolved to:
>
> - **YOLOv8s / YOLO11s** → `in=16, out=1` — the **DFL conv**, a fixed non-learned projection over 16 distribution bins, buried inside the head
> - **YOLO26s** → `in=128, out=14` — the **classification output conv** (YOLO26 removed DFL, so `[-1]` lands somewhere structurally different)
>
> Each would have produced plausible-looking heatmaps from the wrong tensor **without raising an error**, and because they differ per architecture the cross-model comparison would have been meaningless.

Now resolved from `Detect.f` — the feature-map indices the head actually consumes. Architecture-agnostic.

**Expected:** YOLOv8s `Detect at 22, consumes [15, 18, 21]`; YOLO11s/YOLO26s `Detect at 23, consumes [16, 19, 22]`. Different indices, same semantic layer.

**Resolution matters here.** P3 is stride 8 → 64×64 grid at 512px (~8 px/cell). P5 is stride 32 → 16×16 (~32 px/cell), and many Calcification boxes are smaller than one cell — box-overlap metrics on a map that coarse measure nothing.

In [ ]:
probe = {k: YOLO(runs[(k, SEEDS_FOUND[0])]) for k in MODEL_KEYS}

for k, m in probe.items():
    print(f"\n--- {k} ---")
    xai.pick_target_layer(m, scale=C.XAI_SCALE)
    xai.verify_cam_resolution(m, imgsz=C.IMGSZ, scale=C.XAI_SCALE)

### 1b. Scale sanity check — P3 vs P5

Before committing ~45 min to the batch run, confirm P3 actually carries class-discriminative signal.

**What you want:** P3 structured and localised; P5 blobbier. **What would be a problem:** P3 near-uniform or pure noise — that would mean the highest-resolution neck output isn't discriminative for this model, and `XAI_SCALE = "all"` is the fallback.

In [ ]:
import cv2, matplotlib.pyplot as plt, matplotlib.patches as patches
from src.evaluation.metrics import ground_truth_from_yolo

ref = "yolo26s" if "yolo26s" in probe else MODEL_KEYS[0]
gts = ground_truth_from_yolo(TEST_LBL, TEST_IMG)
test_p = sorted(TEST_IMG.glob("*.jpg"))[:3]

fig, axes = plt.subplots(3, 3, figsize=(16, 16))
for row, p in enumerate(test_p):
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    base = cv2.resize(img, (C.IMGSZ, C.IMGSZ))
    axes[row, 0].imshow(base, cmap="gray"); axes[row, 0].set_title("image", fontsize=9)
    for col, sc in enumerate(["p3", "p5"], start=1):
        layer = xai.pick_target_layer(probe[ref], scale=sc, verbose=False)
        cam = xai.compute_cam(probe[ref], img, method="eigencam",
                              target_layer=layer, imgsz=C.IMGSZ)
        ax = axes[row, col]
        ax.imshow(base, cmap="gray"); ax.imshow(cam, cmap="jet", alpha=0.45)
        for _, r in gts[gts.image_id == p.stem].iterrows():
            sx, sy = C.IMGSZ / w, C.IMGSZ / h
            ax.add_patch(patches.Rectangle((r.x1*sx, r.y1*sy), (r.x2-r.x1)*sx,
                                           (r.y2-r.y1)*sy, fill=False,
                                           edgecolor="lime", linewidth=2))
        ebpg = xai.energy_pointing_game(
            cam, gts[gts.image_id == p.stem][["x1","y1","x2","y2"]].to_numpy(),
            (w, h), C.IMGSZ)
        ax.set_title(f"{sc}  EBPG={ebpg:.3f}", fontsize=9)
    for ax in axes[row]:
        ax.axis("off")
plt.suptitle(f"{ref} — P3 (stride 8) vs P5 (stride 32)", y=1.00)
plt.tight_layout()

## 2. Qualitative check (D7)

Fusion runs 1 and 2 both passed `verify()` while producing boxes stacked 8 deep — the cheap visual pass catches what automated gates don't.

In [ ]:
sample = sorted(TEST_IMG.glob("*.jpg"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, p in zip(axes.ravel(), sample):
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    cam = xai.compute_cam(probe[ref], img, method="eigencam", imgsz=C.IMGSZ)
    ax.imshow(cv2.resize(img, (C.IMGSZ, C.IMGSZ)), cmap="gray")
    ax.imshow(cam, cmap="jet", alpha=0.45)
    for _, r in gts[gts.image_id == p.stem].iterrows():
        sx, sy = C.IMGSZ / w, C.IMGSZ / h
        ax.add_patch(patches.Rectangle((r.x1*sx, r.y1*sy), (r.x2-r.x1)*sx,
                                       (r.y2-r.y1)*sy, fill=False,
                                       edgecolor="lime", linewidth=2))
    ax.set_title(p.stem, fontsize=9); ax.axis("off")
plt.suptitle(f"{ref} — EigenCAM @ {C.XAI_SCALE} vs radiologist boxes", y=1.01)
plt.tight_layout()

### 2b. The anchored-class check

Aortic enlargement reaches **0.956** mAP@0.4 with seed std 0.004 — implausibly easy and implausibly stable for a pathology.

**Does the CAM attend to the aorta, or to image centre?** Saliency inside the box → the model learned anatomy. Diffuse or centre-weighted while detection stays correct → position priors, and a concrete demonstration that ~25% of the headline mAP may not require pathological reasoning.

In [ ]:
AORTIC = 0
aortic_imgs = gts[gts.class_id == AORTIC].image_id.unique()[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_id in zip(axes.ravel(), aortic_imgs):
    img = cv2.imread(str(TEST_IMG / f"{img_id}.jpg"), cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    cam = xai.compute_cam(probe[ref], img, method="eigencam", imgsz=C.IMGSZ)
    boxes = gts[(gts.image_id == img_id) & (gts.class_id == AORTIC)][
        ["x1", "y1", "x2", "y2"]].to_numpy()
    ebpg = xai.energy_pointing_game(cam, boxes, (w, h), C.IMGSZ)
    ax.imshow(cv2.resize(img, (C.IMGSZ, C.IMGSZ)), cmap="gray")
    ax.imshow(cam, cmap="jet", alpha=0.45)
    for x1, y1, x2, y2 in boxes:
        sx, sy = C.IMGSZ / w, C.IMGSZ / h
        ax.add_patch(patches.Rectangle((x1*sx, y1*sy), (x2-x1)*sx, (y2-y1)*sy,
                                       fill=False, edgecolor="lime", linewidth=2))
    ax.set_title(f"EBPG={ebpg:.3f}", fontsize=10); ax.axis("off")
plt.suptitle("Aortic enlargement (mAP@0.4 = 0.956) — anatomy or position?", y=1.01)
plt.tight_layout()

## 3. Batch metrics across all runs (D8, ~45 min)

| Metric | What it measures |
|---|---|
| `pointing_game` | CAM argmax inside GT box? Coarse but standard. |
| `ebpg` | Fraction of CAM **energy** inside GT box. **Lead with this** — uses the whole map, so it separates "peaked correctly by luck" from "consistently attends". |
| `cam_box_iou` | IoU of 90th-percentile-thresholded CAM vs GT |

Saved incrementally — an interrupted session doesn't lose completed runs.

In [ ]:
C.RUNS_DIR.mkdir(parents=True, exist_ok=True)
out_csv = C.RUNS_DIR / "xai_metrics.csv"

done, prev = set(), None
if out_csv.exists():
    prev = pd.read_csv(out_csv)
    done = set(map(tuple, prev[["model", "seed", "method"]].drop_duplicates().values))
    print(f"resuming, {len(done)} combos already done")

all_xai = [prev] if prev is not None else []
for (key, seed), w in sorted(runs.items()):
    model = YOLO(w)
    for method in C.XAI_METHODS:
        if (key, seed, method) in done:
            continue
        print(f"\n=== {key} seed={seed} / {method} @ {C.XAI_SCALE} ===")
        df = xai.evaluate_xai(model, TEST_IMG, TEST_LBL, method=method,
                              imgsz=C.IMGSZ, n_images=C.XAI_N_IMAGES,
                              scale=C.XAI_SCALE)
        df["model"], df["seed"] = key, seed
        all_xai.append(df)
        pd.concat(all_xai, ignore_index=True).to_csv(out_csv, index=False)

xai_df = pd.concat(all_xai, ignore_index=True)
print(f"\n{len(xai_df)} rows, {xai_df.groupby(['model','seed']).ngroups} runs")

## 4. Axis A — small vs large targets

**Hypothesis:** explanation faithfulness degrades on small targets faster than mAP does.

**The sharpened version, from the detection results:** YOLO26 detects Calcification 60% better than the others. Does its EBPG on Calcification improve proportionally? If detection improves but faithfulness doesn't, that is decoupling shown on a class where the model demonstrably got better.

In [ ]:
eig = xai_df[xai_df.method == "eigencam"]

grouped = eig.assign(group=eig.class_id.map(
    lambda c: "small" if c in C.SMALL_TARGET_CLASSES
    else ("large" if c in C.LARGE_TARGET_CLASSES else "other")))
per_seed = grouped.groupby(["model", "seed", "group"])["ebpg"].mean().reset_index()
axis_a = per_seed.groupby(["model", "group"])["ebpg"].agg(["mean", "std"]).round(4)
axis_a

In [ ]:
sub = eig.copy()
sub["area_bin"] = pd.qcut(sub.box_area_frac, 5, labels=["XS","S","M","L","XL"])
ps = sub.groupby(["model","seed","area_bin"], observed=False)["ebpg"].mean().reset_index()
pivot = ps.pivot_table(index="area_bin", columns="model", values="ebpg",
                       aggfunc="mean", observed=False)
err = ps.pivot_table(index="area_bin", columns="model", values="ebpg",
                     aggfunc="std", observed=False)
pivot.plot(marker="o", yerr=err, capsize=3, figsize=(8,5),
           title="EBPG vs object size (mean ± std over seeds)")
plt.ylabel("Energy-based pointing game"); plt.grid(alpha=.3)
pivot.round(4)

In [ ]:
# Detection AP beside faithfulness, per class. The paper's core table.
per_class_xai = (eig.groupby(["model","seed","class"])["ebpg"].mean().reset_index()
                 .groupby(["model","class"])["ebpg"].agg(["mean","std"]).round(4))
print(per_class_xai.unstack(0))

print("\nCalcification — detection AP@0.4 was v8 0.079 | v11 0.073 | v26 0.126")
print(per_class_xai.xs("Calcification", level="class") if "Calcification" in
      per_class_xai.index.get_level_values("class") else "n/a")

## 5. Axis B — NMS-free vs NMS

Does YOLO26's NMS-free head produce more faithful explanations? Same seed-noise test applied to detection in notebook 02.

In [ ]:
metrics3 = ["pointing_game", "ebpg", "cam_box_iou"]
per_seed_b = xai_df.groupby(["model","seed","method"])[metrics3].mean().reset_index()
axis_b = per_seed_b.groupby(["model","method"])[metrics3].agg(["mean","std"]).round(4)
print(axis_b)

e = per_seed_b[per_seed_b.method == "eigencam"]
means, stds = e.groupby("model").ebpg.mean(), e.groupby("model").ebpg.std().fillna(0)
gap, noise = means.max() - means.min(), stds.mean()
print(f"\nEBPG gap {gap:.4f} | within-model std {noise:.4f} | ratio {gap/max(noise,1e-9):.1f}x")
print("⚠ inside seed noise -- report as indistinguishable" if gap < noise
      else "⚠ under 2x noise -- hedge the claim" if gap < 2*noise
      else "✓ exceeds 2x noise -- ranking defensible")

## 6. Axis C — accuracy/faithfulness decoupling

Detection ranking (test mAP@0.4): **yolov8s 0.413 > yolo11s 0.402 ≈ yolo26s 0.400**.

If the EBPG ranking differs, accuracy and explanation quality are separate axes — the argument for measuring the second rather than assuming it follows the first.

This survives even if both rankings sit inside seed noise: *"neither accuracy nor faithfulness separates these architectures, but faithfulness varies sharply by lesion size"* is still clean, and Axis A carries it.

In [ ]:
import json

cands = list(Path("/kaggle/input").glob("**/eval_summary.json"))
assert cands, "attach notebook 02's output"
ev = json.loads(cands[0].read_text())

acc = (pd.DataFrame([{"model": v["model"], "seed": v["seed"],
                      "map40": v["map40"], "map50": v["map50"]} for v in ev.values()])
       .groupby("model")[["map40","map50"]].agg(["mean","std"]).round(4))
acc.columns = ["_".join(c) for c in acc.columns]

fai = e.groupby("model")[["ebpg","pointing_game"]].agg(["mean","std"]).round(4)
fai.columns = ["_".join(c) for c in fai.columns]

combined = acc.join(fai)
combined["rank_mAP"] = combined.map40_mean.rank(ascending=False).astype(int)
combined["rank_EBPG"] = combined.ebpg_mean.rank(ascending=False).astype(int)
combined["DECOUPLED"] = combined.rank_mAP != combined.rank_EBPG
combined

## 7. Causal faithfulness — optional, D9 only if ahead

Deletion/insertion AUC needs ~40× the forward passes. **Subsample, seed 0 only.**

A saliency map that looks convincing but has flat deletion AUC isn't explaining the model — the failure MedFocus / MedGround-Bench reported for standard attribution on VinDr.

In [ ]:
RUN_CAUSAL = False   # flip only if D9 is on schedule

if RUN_CAUSAL:
    causal = xai.evaluate_xai(probe[ref], TEST_IMG, TEST_LBL, method="eigencam",
                              imgsz=C.IMGSZ, n_images=50, with_causal=True,
                              causal_subsample=50, scale=C.XAI_SCALE)
    causal.to_csv(C.RUNS_DIR / "xai_causal.csv", index=False)
    print(causal[["deletion_auc", "insertion_auc"]].mean())

## 8. Export for the paper (D10)

In [ ]:
combined.to_csv(C.RUNS_DIR / "table_main.csv")
axis_a.to_csv(C.RUNS_DIR / "table_axis_a_groups.csv")
pivot.to_csv(C.RUNS_DIR / "table_axis_a_bins.csv")
axis_b.to_csv(C.RUNS_DIR / "table_axis_b.csv")
per_class_xai.to_csv(C.RUNS_DIR / "table_per_class_xai.csv")

print("tables written. Write-up reminders:")
print(f"  - CAM taken at {C.XAI_SCALE} (stride 8, 64x64) -- state this; the")
print("    first implementation attributed from the DFL conv and was wrong")
print("  - report mAP@0.4 AND @0.5, TEST split, mean±std over 3 seeds")
print("  - positive-only protocol in the ABSTRACT; published numbers get a")
print("    protocol column and never share a column with ours")
print("  - val/test ranking inversion is itself worth a sentence")
print("  - 2 anchored classes carry ~25% of headline mAP -- report per-class")
print("  - if a gap is inside seed noise, say indistinguishable; don't rank")
print("  - limitations: 512px, s-scale, single fold, positive-only, 40 epochs,")
print("    95.5% of labels from 3 radiologists, WBF@0.25, T4, batch-1 latency")
print("  - Atelectasis n=22 / Pneumothorax n=10: per-class AP with n")